# De-risking Berbasis Drawdown & Universe Lebih Luas — Dua Ide yang Tidak Membantu

**Konteks**: lanjutan dari notebook 23 (volatility targeting — terbukti membantu). Dua ide lain yang
diminta dalam sesi riset yang sama: (2) de-risking halus berbasis drawdown portofolio (bukan
kill-switch biner), dan (3) universe lebih luas dari 8 coin (data 50 coin ada di
`notebooks/data/raw`). Keduanya diuji di notebook ini — **hasilnya negatif untuk keduanya**, ditulis
lebih ringkas dari notebook 23 karena kesimpulannya jelas, bukan karena kurang diuji.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(PROJECT_ROOT)); sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

from data_source.loader import load_symbol
from data_source.fear_greed import load_fear_greed_index
from research.single_position import Panel, BTC
from research.portfolio import run_portfolio, inverse_vol_weights

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

UNIV = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "XRPUSDT", "ADAUSDT", "LINKUSDT", "TRXUSDT", "DOGEUSDT"]
CAPS = {"DOGEUSDT": 0.10}
DEV, TEST = ("2018-07-01", "2022-12-31"), ("2023-01-01", "2026-06-30")
P = Panel({s: load_symbol(s, "1d", "since2018") for s in UNIV}, "2018-01-01", "2026-06-30")
fgi = load_fear_greed_index()["fng_value"].reindex(P.dates).ffill()
btc = P.close[BTC]
market = (btc > btc.rolling(100, min_periods=100).mean()) | (fgi.rolling(14).mean() > 50)
base_w = inverse_vol_weights(P, UNIV, 60, CAPS)
W_V2_60 = base_w.mul(market.astype(float), axis=0) * 0.6
coin_ret = P.close[UNIV].pct_change()
raw_basket_ret = (base_w * coin_ret).sum(axis=1)
vol_exposure_20 = (0.20 / (raw_basket_ret.rolling(20, min_periods=20).std() * np.sqrt(365))).clip(0, 1.0).fillna(0.0)
W_VOLTGT = base_w.mul(vol_exposure_20, axis=0).mul(market.astype(float), axis=0)   # pemenang notebook 23

def stats(eq, info, label):
    years = (eq.index[-1] - eq.index[0]).days / 365.25
    tot = eq.iloc[-1] / eq.iloc[0] - 1
    dr = eq.pct_change().dropna()
    dd = (eq / eq.cummax() - 1).min()
    return dict(strategi=label, profit=tot, cagr=(1 + tot) ** (1 / years) - 1, max_dd=dd,
                sharpe=dr.mean() / dr.std() * np.sqrt(365) if dr.std() > 0 else np.nan,
                rugi_harian_terburuk=dr.min(), pct_hari_rugi_gt2pct=(dr < -0.02).mean(), n_rebalance=info["n_rebalance"])

def show(res):
    out = res.copy().astype(object)
    for c in ("profit", "cagr", "max_dd", "rugi_harian_terburuk"):
        out[c] = res[c].map(lambda v: f"{v:+.1%}")
    out["pct_hari_rugi_gt2pct"] = res["pct_hari_rugi_gt2pct"].map(lambda v: f"{v:.1%}")
    out["sharpe"] = res["sharpe"].map(lambda v: f"{v:.2f}")
    return out

print("Setup selesai.")

Setup selesai.


## Ide #2 — De-risking halus berbasis drawdown portofolio

Konsep: bukan kill-switch biner (berhenti total di satu ambang rugi), tapi eksposur berkurang
**bertahap linear** begitu portofolio sedang drawdown — makin dalam drawdown-nya, makin kecil
eksposurnya, sampai 0% di ambang terdalam. Sinyal drawdown diukur dari basket inverse-vol tanpa
skala (`ref_dd`) biar tidak ada feedback loop (eksposur mempengaruhi drawdown mempengaruhi eksposur).

In [2]:
ref_equity = (1 + raw_basket_ret.fillna(0)).cumprod()
ref_dd = ref_equity / ref_equity.cummax() - 1

def derisk_weights(x_pct, y_pct, base_exposure=0.6):
    scale = ((ref_dd + y_pct) / (y_pct - x_pct)).clip(0, 1)   # 1 di atas -x%, turun linear ke 0 di -y%
    return base_w.mul(scale, axis=0).mul(market.astype(float), axis=0) * base_exposure

print("DEV: sweep ambang de-risking (standalone, menggantikan 60% tetap)\n")
rows = [stats(*run_portfolio(P, W_V2_60, *DEV), "V2-60 baseline")]
for x, y in [(5, 20), (10, 20), (10, 30), (15, 30), (10, 40), (15, 40), (20, 40), (20, 50)]:
    eq, info = run_portfolio(P, derisk_weights(x / 100, y / 100), *DEV, freq="M")
    rows.append(stats(eq, info, f"derisk {x}%->{y}%"))
show(pd.DataFrame(rows).set_index("strategi"))

DEV: sweep ambang de-risking (standalone, menggantikan 60% tetap)



,profit,cagr,max_dd,sharpe,rugi_harian_terburuk,pct_hari_rugi_gt2pct,n_rebalance
strategi,,,,,,,
V2-60 baseline,+1477.7%,+84.6%,-29.4%,1.65,-10.7%,7.3%,96
derisk 5%->20%,+476.7%,+47.6%,-20.3%,1.35,-10.7%,2.7%,87
derisk 10%->20%,+487.6%,+48.2%,-20.3%,1.33,-10.7%,2.9%,87
derisk 10%->30%,+523.7%,+50.2%,-20.2%,1.34,-10.7%,3.5%,87
derisk 15%->30%,+521.1%,+50.0%,-20.2%,1.32,-10.7%,3.7%,87
derisk 10%->40%,+616.8%,+54.9%,-20.2%,1.40,-10.7%,4.0%,92
derisk 15%->40%,+630.8%,+55.6%,-20.9%,1.40,-10.7%,4.4%,92
derisk 20%->40%,+670.3%,+57.4%,-22.8%,1.41,-10.7%,4.6%,92
derisk 20%->50%,+738.7%,+60.4%,-25.2%,1.44,-10.7%,4.9%,97


**Standalone sudah lemah**: Sharpe 1.32-1.44, semua di bawah baseline (1.65) — dan rugi harian
terburuk **persis sama** dengan baseline (−10.7%) di SETIAP konfigurasi. Ini bukan kebetulan: sinyal
drawdown baru terbentuk SETELAH kerugian terjadi, jadi tidak bisa mencegah hari pertama yang jatuh
tiba-tiba (beda dari vol-targeting yang bereaksi ke volatilitas naik, yang biasanya muncul duluan).

In [3]:
print("Apa NAMBAH manfaat di ATAS vol-targeting tv=20% (pemenang notebook 23)? (DEV)\n")
rows2 = [stats(*run_portfolio(P, W_V2_60, *DEV), "V2-60 baseline"),
         stats(*run_portfolio(P, W_VOLTGT, *DEV, freq="M"), "voltgt tv=20% saja (nb23)")]
for x, y in [(10, 30), (15, 40), (20, 50)]:
    scale = ((ref_dd + y / 100) / (y / 100 - x / 100)).clip(0, 1)
    w_combo = base_w.mul(vol_exposure_20, axis=0).mul(scale, axis=0).mul(market.astype(float), axis=0)
    eq, info = run_portfolio(P, w_combo, *DEV, freq="M")
    rows2.append(stats(eq, info, f"voltgt tv=20% + derisk {x}%->{y}%"))
show(pd.DataFrame(rows2).set_index("strategi"))

Apa NAMBAH manfaat di ATAS vol-targeting tv=20% (pemenang notebook 23)? (DEV)



,profit,cagr,max_dd,sharpe,rugi_harian_terburuk,pct_hari_rugi_gt2pct,n_rebalance
strategi,,,,,,,
V2-60 baseline,+1477.7%,+84.6%,-29.4%,1.65,-10.7%,7.3%,96
voltgt tv=20% saja (nb23),+394.7%,+42.6%,-14.9%,1.68,-7.6%,3.2%,96
voltgt tv=20% + derisk 10%->30%,+170.1%,+24.7%,-12.2%,1.31,-7.6%,1.8%,87
voltgt tv=20% + derisk 15%->40%,+188.1%,+26.5%,-12.2%,1.36,-7.6%,1.9%,92
voltgt tv=20% + derisk 20%->50%,+211.7%,+28.7%,-12.2%,1.42,-7.6%,2.0%,97


**Dikombinasikan juga tidak membantu** — malah menurunkan Sharpe (1.68→1.31-1.42) dan profit
secara signifikan, sementara max_dd cuma membaik tipis (−14.9%→−12.2%). Vol-targeting sendirian
sudah mengerjakan tugas pengurangan eksposur dengan lebih baik; menambah lapisan de-risking kedua di
atasnya jadi redundan dan net-negatif. **Ide #2 tidak dipakai.**

## Ide #3 — Universe lebih luas (bukan cuma 8 coin)

Pakai 12 coin tambahan dari `notebooks/data/raw` yang datanya tersedia dari 2021-01 (SOL, AVAX, DOT,
UNI, NEAR, ALGO, ETC, HBAR, AAVE, XLM, SHIB, ICP) — total 20 coin. Karena datanya cuma dari 2021,
periode dev untuk bagian ini **beda** dari notebook lain: dev2 = 2021-01→2022-12 (tetap mencakup
crash 2022), uji tetap 2023-01→2026-06 (sama, supaya tetap bisa dibandingkan).

In [4]:
EXTRA = ["SOLUSDT", "AVAXUSDT", "DOTUSDT", "UNIUSDT", "NEARUSDT", "ALGOUSDT", "ETCUSDT",
         "HBARUSDT", "AAVEUSDT", "XLMUSDT", "SHIBUSDT", "ICPUSDT"]
UNIV20 = UNIV + EXTRA
CAPS20 = {**CAPS, "SHIBUSDT": 0.10}
DEV2 = ("2021-01-01", "2022-12-31")

def load_full(sym):
    df = pd.read_csv(PROJECT_ROOT / "notebooks" / "data" / "raw" / f"{sym}_1d_full.csv", parse_dates=[0], index_col=0)
    df.columns = [c.lower() for c in df.columns]
    return df

ohlc20 = {s: load_symbol(s, "1d", "since2018") for s in UNIV}
ohlc20.update({s: load_full(s) for s in EXTRA})
P20 = Panel(ohlc20, "2018-01-01", "2026-06-30")
market20 = market.reindex(P20.dates)

ret20 = P20.close[UNIV20].pct_change()
avg_corr = lambda yr: (lambda c: (c.values.sum() - len(UNIV20)) / (len(UNIV20) * (len(UNIV20) - 1)))(
    ret20.loc[f"{yr}-01-01":f"{yr}-12-31"].corr())
print(f"Rata-rata korelasi pairwise 20 coin -- 2022 (bear): {avg_corr(2022):.2f} | 2024 (bull): {avg_corr(2024):.2f}")

bw8_dev2 = inverse_vol_weights(P20, UNIV, 60, CAPS).mul(market20.astype(float), axis=0) * 0.6
bw20 = inverse_vol_weights(P20, UNIV20, 60, CAPS20).mul(market20.astype(float), axis=0) * 0.6

print("\nDEV2 (2021-01..2022-12):")
rows = [stats(*run_portfolio(P20, bw8_dev2, *DEV2, freq="M"), "V2-60 (8 coin)"),
        stats(*run_portfolio(P20, bw20, *DEV2, freq="M"), "V-20 (20 coin)")]
display(show(pd.DataFrame(rows).set_index("strategi")))

print("\nUJI (2023-2026):")
rows_t = [stats(*run_portfolio(P20, bw8_dev2, *TEST, freq="M"), "V2-60 (8 coin)"),
          stats(*run_portfolio(P20, bw20, *TEST, freq="M"), "V-20 (20 coin)")]
show(pd.DataFrame(rows_t).set_index("strategi"))

Rata-rata korelasi pairwise 20 coin -- 2022 (bear): 0.72 | 2024 (bull): 0.51

DEV2 (2021-01..2022-12):


,profit,cagr,max_dd,sharpe,rugi_harian_terburuk,pct_hari_rugi_gt2pct,n_rebalance
strategi,,,,,,,
V2-60 (8 coin),+482.4%,+141.8%,-23.8%,1.98,-10.7%,7.7%,41
V-20 (20 coin),+618.9%,+168.7%,-23.8%,2.21,-10.7%,7.4%,42



UJI (2023-2026):


,profit,cagr,max_dd,sharpe,rugi_harian_terburuk,pct_hari_rugi_gt2pct,n_rebalance
strategi,,,,,,,
V2-60 (8 coin),+156.7%,+31.0%,-21.3%,1.17,-7.6%,5.2%,74
V-20 (20 coin),+145.7%,+29.3%,-29.4%,0.98,-10.6%,7.4%,74


**Korelasi pairwise rata-rata 20 coin: 0.72 pas crash 2022, 0.51 pas bull 2024** — mengkonfirmasi
diagnosis notebook 14 (korelasi naik tajam pas market jatuh) berlaku luas, bukan cuma di 8 coin awal.
Hasil backtest konsisten dengan ini: **di uji 2023-2026, 20 coin lebih buruk dari 8 coin** di semua
metrik — profit lebih rendah, max_dd lebih dalam (−29.4% vs −21.3%), Sharpe lebih rendah (0.98 vs
1.17). Memperluas universe coin crypto large/mid-cap **tidak menambah diversifikasi riil** karena
semuanya didorong oleh beta crypto yang sama, apalagi pas krisis — persis waktu diversifikasi paling
dibutuhkan. **Ide #3 tidak dipakai.**

## Kesimpulan

Dua dari tiga ide yang diminta **tidak menambah nilai**:
- **De-risking berbasis drawdown**: terlalu lambat (reaktif, bukan prediktif) — rugi harian terburuk
  tidak berubah sama sekali di semua konfigurasi, dan dikombinasikan dengan vol-targeting malah
  menurunkan Sharpe tanpa perbaikan drawdown yang berarti.
- **Universe lebih luas (20 coin)**: korelasi pairwise rata-rata tetap tinggi (0.72) pas crash —
  "diversifikasi" dari menambah coin crypto large/mid-cap itu ilusi, karena semua bergerak bareng
  justru pas paling dibutuhkan. Terbukti lebih buruk di uji out-of-sample di semua metrik.

**Hanya ide #1 (volatility targeting, notebook 23) yang terbukti membantu** dari ketiga ide yang
diminta di sesi ini. Kombinasi final yang direkomendasikan: **V2-60 (8 coin, inverse-vol, filter H8b)
+ volatility targeting (target 20%/tahun, lookback 20 hari, max eksposur 100%)** — tidak perlu
ditambah de-risking drawdown atau universe lebih luas.